In [ ]:
# ============================================================
# 🎧 DSP AUDIO SEPARATION STUDIO
# FFT + FIR + IIR
# Gradio 6.x
# ============================================================
#
# PROJECT:
# Speech and Music Separation using FFT, FIR and IIR Filters
#
# INPUT:
# Mixed audio = Speech + Music
#
# OUTPUT:
# Speech and Music separated using:
#   1. FFT frequency-domain separation
#   2. FIR low-pass/high-pass filtering
#   3. IIR Butterworth low-pass/high-pass filtering
#
# ANALYSIS:
#   • Individual waveforms
#   • Individual frequency spectra
#   • Overall waveform comparison
#   • Overall spectrum comparison
#   • Correlation
#   • SNR
#   • RMSE
# ============================================================


# ============================================================
# 1. INSTALL REQUIRED LIBRARIES
# ============================================================

!pip -q install "gradio>=6,<7" numpy scipy matplotlib soundfile pandas


# ============================================================
# 2. IMPORT LIBRARIES
# ============================================================

import os
import gc
import numpy as np
import pandas as pd
import soundfile as sf
import matplotlib.pyplot as plt
import gradio as gr

from scipy import signal
from scipy.signal import firwin, butter, lfilter, sosfilt


# ============================================================
# 3. BACKGROUND IMAGE
# ============================================================

BACKGROUND_URL = "https://i.ibb.co/9mMML0H9/background.png"


# ============================================================
# 4. AUDIO LOADING
# ============================================================

def load_audio(filename):
    """
    Load WAV audio.

    Steps:
    1. Read audio file.
    2. Convert stereo audio to mono.
    3. Convert samples to float64.
    4. Remove DC component.
    """

    audio, fs = sf.read(filename)

    # Stereo → Mono
    if audio.ndim > 1:
        audio = np.mean(audio, axis=1)

    audio = audio.astype(np.float64)

    # Remove DC component
    audio = audio - np.mean(audio)

    return audio, fs


# ============================================================
# 5. AUDIO NORMALIZATION
# ============================================================

def normalize_audio(audio, peak=0.95):
    """
    Normalize audio to avoid clipping during playback.
    """

    maximum = np.max(np.abs(audio))

    if maximum == 0:
        return audio

    return audio / maximum * peak


# ============================================================
# 6. PERFORMANCE METRICS
# ============================================================

def correlation_coefficient(reference, estimated):
    """
    Measures similarity between reference and separated signal.

    Value closer to 1:
        Higher similarity.

    Value closer to 0:
        Lower similarity.
    """

    reference = reference - np.mean(reference)
    estimated = estimated - np.mean(estimated)

    denominator = np.sqrt(
        np.sum(reference ** 2) *
        np.sum(estimated ** 2)
    )

    if denominator == 0:
        return 0

    return np.sum(reference * estimated) / denominator


def calculate_snr(reference, estimated):
    """
    Calculates Signal-to-Noise Ratio.

    Higher SNR generally indicates
    lower separation error.
    """

    scale = np.dot(
        reference,
        estimated
    ) / (
        np.dot(
            estimated,
            estimated
        ) + 1e-12
    )

    estimated_scaled = scale * estimated

    error = reference - estimated_scaled

    signal_power = np.mean(
        reference ** 2
    )

    noise_power = np.mean(
        error ** 2
    )

    if noise_power == 0:
        return np.inf

    return 10 * np.log10(
        signal_power / noise_power
    )


def calculate_rmse(reference, estimated):
    """
    Calculates Root Mean Square Error.

    Lower RMSE generally indicates
    smaller separation error.
    """

    scale = np.dot(
        reference,
        estimated
    ) / (
        np.dot(
            estimated,
            estimated
        ) + 1e-12
    )

    estimated_scaled = scale * estimated

    return np.sqrt(
        np.mean(
            (reference - estimated_scaled) ** 2
        )
    )


# ============================================================
# 7. FFT SEPARATION
# ============================================================

def fft_separation(audio, fs, cutoff):
    """
    FFT-based frequency-domain separation.

    Low-frequency components:
        → Speech

    High-frequency components:
        → Music

    A binary frequency mask is used around
    the selected cutoff frequency.
    """

    N = len(audio)

    # Forward FFT
    X = np.fft.rfft(audio)

    frequencies = np.fft.rfftfreq(
        N,
        d=1 / fs
    )

    # Low frequencies → Speech
    speech_mask = frequencies <= cutoff

    # High frequencies → Music
    music_mask = frequencies > cutoff

    speech_fft = X * speech_mask

    music_fft = X * music_mask

    # Inverse FFT
    speech = np.fft.irfft(
        speech_fft,
        n=N
    )

    music = np.fft.irfft(
        music_fft,
        n=N
    )

    return speech, music


# ============================================================
# 8. FIR SEPARATION
# ============================================================

def fir_separation(
    audio,
    fs,
    cutoff,
    taps
):
    """
    FIR filtering.

    Low-pass FIR:
        → Speech

    High-pass FIR:
        → Music

    Hamming window is used for FIR design.
    """

    taps = int(taps)

    # Low-pass FIR
    low_filter = firwin(
        numtaps=taps,
        cutoff=cutoff,
        fs=fs,
        window="hamming",
        pass_zero="lowpass"
    )

    # High-pass FIR
    high_filter = firwin(
        numtaps=taps,
        cutoff=cutoff,
        fs=fs,
        window="hamming",
        pass_zero="highpass"
    )

    # Apply low-pass filter
    speech = lfilter(
        low_filter,
        [1.0],
        audio
    )

    # Apply high-pass filter
    music = lfilter(
        high_filter,
        [1.0],
        audio
    )

    return speech, music


# ============================================================
# 9. IIR BUTTERWORTH SEPARATION
# ============================================================

def iir_separation(
    audio,
    fs,
    cutoff,
    order
):
    """
    IIR Butterworth filtering.

    Low-pass Butterworth:
        → Speech

    High-pass Butterworth:
        → Music
    """

    order = int(order)

    # Low-pass Butterworth
    low_filter = butter(
        order,
        cutoff,
        btype="lowpass",
        fs=fs,
        output="sos"
    )

    # High-pass Butterworth
    high_filter = butter(
        order,
        cutoff,
        btype="highpass",
        fs=fs,
        output="sos"
    )

    # Apply filters
    speech = sosfilt(
        low_filter,
        audio
    )

    music = sosfilt(
        high_filter,
        audio
    )

    return speech, music


# ============================================================
# 10. WAVEFORM PLOT FUNCTION
# ============================================================

def create_waveform_plot(
    audio,
    fs,
    title,
    filename
):
    """
    Creates an individual time-domain waveform.

    The first 30 seconds are displayed to keep
    visualization memory efficient.

    X-axis:
        Time in seconds

    Y-axis:
        Signal amplitude
    """

    duration = min(
        30,
        len(audio) / fs
    )

    samples = int(
        duration * fs
    )

    # Reduce number of displayed points
    step = max(
        1,
        samples // 5000
    )

    time = (
        np.arange(samples)[::step]
        / fs
    )

    fig = plt.figure(
        figsize=(10, 4.5),
        dpi=120
    )

    plt.plot(
        time,
        audio[:samples:step],
        linewidth=0.8
    )

    plt.xlabel(
        "Time (seconds)"
    )

    plt.ylabel(
        "Amplitude"
    )

    plt.title(
        title,
        fontweight="bold"
    )

    plt.grid(
        alpha=0.25
    )

    plt.tight_layout()

    plt.savefig(
        filename,
        dpi=140,
        bbox_inches="tight"
    )

    plt.close(fig)

    return filename


# ============================================================
# 11. SPECTRUM PLOT FUNCTION
# ============================================================

def create_spectrum_plot(
    audio,
    fs,
    title,
    filename,
    cutoff=None
):
    """
    Creates an individual frequency-domain spectrum.

    FFT is calculated using a Hanning window.

    Frequency range displayed:
        0 to 8000 Hz

    X-axis:
        Frequency in Hz

    Y-axis:
        Magnitude
    """

    # Use limited samples for visualization
    max_samples = min(
        len(audio),
        int(60 * fs),
        262144
    )

    x = audio[:max_samples]

    N = len(x)

    # Hanning window reduces spectral leakage
    window = np.hanning(N)

    # FFT
    X = np.fft.rfft(
        x * window
    )

    frequencies = np.fft.rfftfreq(
        N,
        d=1 / fs
    )

    magnitude = (
        np.abs(X) / N
    )

    # Display only up to 8 kHz
    mask = frequencies <= 8000

    fig = plt.figure(
        figsize=(10, 4.5),
        dpi=120
    )

    plt.plot(
        frequencies[mask],
        magnitude[mask],
        linewidth=0.8
    )

    # Show cutoff frequency when required
    if cutoff is not None:

        plt.axvline(
            cutoff,
            linestyle="--",
            linewidth=2,
            label=f"Cutoff = {int(cutoff)} Hz"
        )

        plt.legend()

    plt.xlabel(
        "Frequency (Hz)"
    )

    plt.ylabel(
        "Magnitude"
    )

    plt.title(
        title,
        fontweight="bold"
    )

    plt.grid(
        alpha=0.25
    )

    plt.tight_layout()

    plt.savefig(
        filename,
        dpi=140,
        bbox_inches="tight"
    )

    plt.close(fig)

    return filename


# ============================================================
# 12. OVERALL WAVEFORM COMPARISON
# ============================================================

def create_waveform_comparison(
    mixed,
    fft_speech,
    fft_music,
    fir_speech,
    fir_music,
    iir_speech,
    iir_music,
    fs
):
    """
    Overall waveform comparison.

    Shows the separated speech/music waveforms
    from FFT, FIR and IIR methods.

    This graph is useful for visually comparing
    the time-domain behavior of all methods.
    """

    duration = min(
        30,
        len(mixed) / fs
    )

    samples = int(
        duration * fs
    )

    step = max(
        1,
        samples // 5000
    )

    time = (
        np.arange(samples)[::step]
        / fs
    )

    fig = plt.figure(
        figsize=(12, 6),
        dpi=120
    )

    plt.plot(
        time,
        mixed[:samples:step],
        label="Mixed Input",
        linewidth=0.8
    )

    plt.plot(
        time,
        fft_speech[:samples:step],
        label="FFT Speech",
        linewidth=0.7
    )

    plt.plot(
        time,
        fft_music[:samples:step],
        label="FFT Music",
        linewidth=0.7
    )

    plt.plot(
        time,
        fir_speech[:samples:step],
        label="FIR Speech",
        linewidth=0.7
    )

    plt.plot(
        time,
        fir_music[:samples:step],
        label="FIR Music",
        linewidth=0.7
    )

    plt.plot(
        time,
        iir_speech[:samples:step],
        label="IIR Speech",
        linewidth=0.7
    )

    plt.plot(
        time,
        iir_music[:samples:step],
        label="IIR Music",
        linewidth=0.7
    )

    plt.xlabel(
        "Time (seconds)"
    )

    plt.ylabel(
        "Amplitude"
    )

    plt.title(
        "Overall Waveform Comparison",
        fontweight="bold"
    )

    plt.legend(
        fontsize=8,
        ncol=2
    )

    plt.grid(
        alpha=0.25
    )

    plt.tight_layout()

    filename = "Overall_Waveform_Comparison.png"

    plt.savefig(
        filename,
        dpi=140,
        bbox_inches="tight"
    )

    plt.close(fig)

    return filename


# ============================================================
# 13. OVERALL SPECTRUM COMPARISON
# ============================================================

def create_spectrum_comparison(
    mixed,
    fft_speech,
    fft_music,
    fir_speech,
    fir_music,
    iir_speech,
    iir_music,
    fs,
    cutoff
):
    """
    Overall frequency-domain comparison.

    All important separated signals are shown
    together in one frequency-domain graph.

    The vertical dashed line represents the
    selected separation cutoff.
    """

    signals = {
        "Mixed Input": mixed,
        "FFT Speech": fft_speech,
        "FFT Music": fft_music,
        "FIR Speech": fir_speech,
        "FIR Music": fir_music,
        "IIR Speech": iir_speech,
        "IIR Music": iir_music
    }

    fig = plt.figure(
        figsize=(12, 6),
        dpi=120
    )

    for name, audio in signals.items():

        max_samples = min(
            len(audio),
            int(60 * fs),
            262144
        )

        x = audio[:max_samples]

        N = len(x)

        window = np.hanning(N)

        X = np.fft.rfft(
            x * window
        )

        frequencies = np.fft.rfftfreq(
            N,
            d=1 / fs
        )

        magnitude = (
            np.abs(X) / N
        )

        mask = frequencies <= 8000

        plt.plot(
            frequencies[mask],
            magnitude[mask],
            linewidth=0.7,
            label=name
        )

    plt.axvline(
        cutoff,
        linestyle="--",
        linewidth=2,
        label=f"Cutoff = {int(cutoff)} Hz"
    )

    plt.xlabel(
        "Frequency (Hz)"
    )

    plt.ylabel(
        "Magnitude"
    )

    plt.title(
        "Overall FFT Spectrum Comparison",
        fontweight="bold"
    )

    plt.legend(
        fontsize=8,
        ncol=2
    )

    plt.grid(
        alpha=0.25
    )

    plt.tight_layout()

    filename = "Overall_Spectrum_Comparison.png"

    plt.savefig(
        filename,
        dpi=140,
        bbox_inches="tight"
    )

    plt.close(fig)

    return filename


# ============================================================
# 14. MAIN PROCESSING FUNCTION
# ============================================================

def process_audio(
    mixed_file,
    speech_reference,
    music_reference,
    cutoff,
    fir_taps,
    fir_order,
    iir_order
):

    if mixed_file is None:

        raise gr.Error(
            "Please upload the mixed audio file first."
        )


    # ========================================================
    # LOAD MIXED AUDIO
    # ========================================================

    mixed, fs = load_audio(
        mixed_file
    )

    samples = len(mixed)

    duration = samples / fs


    # ========================================================
    # FFT SEPARATION
    # ========================================================

    fft_speech, fft_music = fft_separation(
        mixed,
        fs,
        cutoff
    )


    # ========================================================
    # FIR SEPARATION
    # ========================================================

    fir_speech, fir_music = fir_separation(
        mixed,
        fs,
        cutoff,
        fir_taps
    )


    # ========================================================
    # IIR SEPARATION
    # ========================================================

    iir_speech, iir_music = iir_separation(
        mixed,
        fs,
        cutoff,
        iir_order
    )


    # ========================================================
    # NORMALIZE OUTPUT SIGNALS
    # ========================================================

    fft_speech = normalize_audio(
        fft_speech
    )

    fft_music = normalize_audio(
        fft_music
    )

    fir_speech = normalize_audio(
        fir_speech
    )

    fir_music = normalize_audio(
        fir_music
    )

    iir_speech = normalize_audio(
        iir_speech
    )

    iir_music = normalize_audio(
        iir_music
    )


    # ========================================================
    # SAVE SEPARATED AUDIO FILES
    # ========================================================

    sf.write(
        "FFT_Speech.wav",
        fft_speech,
        fs
    )

    sf.write(
        "FFT_Music.wav",
        fft_music,
        fs
    )

    sf.write(
        "FIR_Speech.wav",
        fir_speech,
        fs
    )

    sf.write(
        "FIR_Music.wav",
        fir_music,
        fs
    )

    sf.write(
        "IIR_Speech.wav",
        iir_speech,
        fs
    )

    sf.write(
        "IIR_Music.wav",
        iir_music,
        fs
    )


    # ========================================================
    # CREATE INDIVIDUAL WAVEFORM GRAPHS
    # ========================================================
    #
    # These graphs show the time-domain behavior of every
    # important input/output signal.
    # ========================================================

    mixed_waveform = create_waveform_plot(
        mixed,
        fs,
        "Mixed Input — Speech + Music",
        "Mixed_Waveform.png"
    )

    fft_speech_waveform = create_waveform_plot(
        fft_speech,
        fs,
        "FFT Separated Speech — Waveform",
        "FFT_Speech_Waveform.png"
    )

    fft_music_waveform = create_waveform_plot(
        fft_music,
        fs,
        "FFT Separated Music — Waveform",
        "FFT_Music_Waveform.png"
    )

    fir_speech_waveform = create_waveform_plot(
        fir_speech,
        fs,
        "FIR Separated Speech — Waveform",
        "FIR_Speech_Waveform.png"
    )

    fir_music_waveform = create_waveform_plot(
        fir_music,
        fs,
        "FIR Separated Music — Waveform",
        "FIR_Music_Waveform.png"
    )

    iir_speech_waveform = create_waveform_plot(
        iir_speech,
        fs,
        "IIR Separated Speech — Waveform",
        "IIR_Speech_Waveform.png"
    )

    iir_music_waveform = create_waveform_plot(
        iir_music,
        fs,
        "IIR Separated Music — Waveform",
        "IIR_Music_Waveform.png"
    )


    # ========================================================
    # CREATE INDIVIDUAL SPECTRUM GRAPHS
    # ========================================================
    #
    # These graphs show the frequency-domain characteristics
    # of every important signal.
    #
    # Frequency range is limited to 8 kHz for clear display.
    # ========================================================

    mixed_spectrum = create_spectrum_plot(
        mixed,
        fs,
        "Mixed Input — Frequency Spectrum",
        "Mixed_Spectrum.png",
        cutoff
    )

    fft_speech_spectrum = create_spectrum_plot(
        fft_speech,
        fs,
        "FFT Separated Speech — Spectrum",
        "FFT_Speech_Spectrum.png",
        cutoff
    )

    fft_music_spectrum = create_spectrum_plot(
        fft_music,
        fs,
        "FFT Separated Music — Spectrum",
        "FFT_Music_Spectrum.png",
        cutoff
    )

    fir_speech_spectrum = create_spectrum_plot(
        fir_speech,
        fs,
        "FIR Separated Speech — Spectrum",
        "FIR_Speech_Spectrum.png",
        cutoff
    )

    fir_music_spectrum = create_spectrum_plot(
        fir_music,
        fs,
        "FIR Separated Music — Spectrum",
        "FIR_Music_Spectrum.png",
        cutoff
    )

    iir_speech_spectrum = create_spectrum_plot(
        iir_speech,
        fs,
        "IIR Separated Speech — Spectrum",
        "IIR_Speech_Spectrum.png",
        cutoff
    )

    iir_music_spectrum = create_spectrum_plot(
        iir_music,
        fs,
        "IIR Separated Music — Spectrum",
        "IIR_Music_Spectrum.png",
        cutoff
    )


    # ========================================================
    # OVERALL ANALYSIS GRAPHS
    # ========================================================

    overall_waveform = create_waveform_comparison(
        mixed,
        fft_speech,
        fft_music,
        fir_speech,
        fir_music,
        iir_speech,
        iir_music,
        fs
    )

    overall_spectrum = create_spectrum_comparison(
        mixed,
        fft_speech,
        fft_music,
        fir_speech,
        fir_music,
        iir_speech,
        iir_music,
        fs,
        cutoff
    )


    # ========================================================
    # PERFORMANCE EVALUATION
    # ========================================================
    #
    # Reference speech/music files are used ONLY for evaluation.
    #
    # They are NOT used as input to the separation process.
    # ========================================================

    rows = []

    if (
        speech_reference is not None
        and music_reference is not None
    ):

        speech_ref, speech_fs = load_audio(
            speech_reference
        )

        music_ref, music_fs = load_audio(
            music_reference
        )


        # ----------------------------------------------------
        # Match sampling frequency if necessary
        # ----------------------------------------------------

        if speech_fs != fs:

            speech_ref = signal.resample_poly(
                speech_ref,
                fs,
                speech_fs
            )


        if music_fs != fs:

            music_ref = signal.resample_poly(
                music_ref,
                fs,
                music_fs
            )


        # ----------------------------------------------------
        # Match signal lengths
        # ----------------------------------------------------

        common_length = min(

            len(speech_ref),

            len(music_ref),

            len(fft_speech),

            len(fft_music),

            len(fir_speech),

            len(fir_music),

            len(iir_speech),

            len(iir_music)

        )


        speech_ref = speech_ref[
            :common_length
        ]

        music_ref = music_ref[
            :common_length
        ]


        # ----------------------------------------------------
        # Compare all three separation methods
        # ----------------------------------------------------

        methods = [

            (
                "FFT",
                fft_speech[:common_length],
                fft_music[:common_length]
            ),

            (
                "FIR",
                fir_speech[:common_length],
                fir_music[:common_length]
            ),

            (
                "IIR",
                iir_speech[:common_length],
                iir_music[:common_length]
            )

        ]


        for name, speech_est, music_est in methods:

            rows.append([

                name,

                round(
                    correlation_coefficient(
                        speech_ref,
                        speech_est
                    ),
                    4
                ),

                round(
                    correlation_coefficient(
                        music_ref,
                        music_est
                    ),
                    4
                ),

                round(
                    calculate_snr(
                        speech_ref,
                        speech_est
                    ),
                    2
                ),

                round(
                    calculate_snr(
                        music_ref,
                        music_est
                    ),
                    2
                ),

                round(
                    calculate_rmse(
                        speech_ref,
                        speech_est
                    ),
                    5
                ),

                round(
                    calculate_rmse(
                        music_ref,
                        music_est
                    ),
                    5
                )

            ])


    # ========================================================
    # PERFORMANCE DATAFRAME
    # ========================================================

    results = pd.DataFrame(
        rows,
        columns=[
            "Method",
            "Speech Correlation",
            "Music Correlation",
            "Speech SNR (dB)",
            "Music SNR (dB)",
            "Speech RMSE",
            "Music RMSE"
        ]
    )


    # ========================================================
    # RESULT INFORMATION
    # ========================================================

    fir_delay = (
        int(fir_taps) - 1
    ) // 2


    result_html = f"""

    <div class="result-title">
        ✅ PROCESSING COMPLETED
    </div>

    <div class="result-subtitle">
        Audio separation and DSP analysis have been completed successfully.
    </div>


    <div class="stats-grid">

        <div class="stat-card">
            <div class="stat-label">
                SAMPLING RATE
            </div>

            <div class="stat-value">
                {fs:,} Hz
            </div>
        </div>


        <div class="stat-card">
            <div class="stat-label">
                SAMPLES
            </div>

            <div class="stat-value">
                {samples:,}
            </div>
        </div>


        <div class="stat-card">
            <div class="stat-label">
                DURATION
            </div>

            <div class="stat-value">
                {duration:.2f} sec
            </div>
        </div>


        <div class="stat-card">
            <div class="stat-label">
                CUTOFF
            </div>

            <div class="stat-value">
                {int(cutoff)} Hz
            </div>
        </div>

    </div>


    <div class="result-heading">
        ⚙️ DSP CONFIGURATION
    </div>

    <div class="config-row">

        <span>
            FIR Taps
            <b>{int(fir_taps)}</b>
        </span>

        <span>
            FIR Order
            <b>{int(fir_order)}</b>
        </span>

        <span>
            IIR Order
            <b>{int(iir_order)}</b>
        </span>

        <span>
            FIR Delay
            <b>{fir_delay} samples</b>
        </span>

    </div>


    <div class="result-heading">
        🔬 SEPARATION METHODS
    </div>

    <div class="method-row">

        <div class="method-item fft-method">
            <span class="method-icon">🟣</span>
            <b>FFT</b>
            <small>
                Frequency-domain separation
            </small>
        </div>


        <div class="method-item fir-method">
            <span class="method-icon">💜</span>
            <b>FIR</b>
            <small>
                Low-pass Speech / High-pass Music
            </small>
        </div>


        <div class="method-item iir-method">
            <span class="method-icon">🔵</span>
            <b>IIR</b>
            <small>
                Butterworth filtering
            </small>
        </div>

    </div>


    <div class="success-box">
        🟢 <b>STATUS:</b>
        Separation and analysis completed successfully.
    </div>

    """


    # ========================================================
    # MEMORY CLEANUP
    # ========================================================

    gc.collect()


    # ========================================================
    # RETURN ALL OUTPUTS
    # ========================================================

    return (

        result_html,

        # Audio outputs
        "FFT_Speech.wav",
        "FFT_Music.wav",

        "FIR_Speech.wav",
        "FIR_Music.wav",

        "IIR_Speech.wav",
        "IIR_Music.wav",

        # Individual waveforms
        mixed_waveform,
        fft_speech_waveform,
        fft_music_waveform,

        fir_speech_waveform,
        fir_music_waveform,

        iir_speech_waveform,
        iir_music_waveform,

        # Individual spectra
        mixed_spectrum,
        fft_speech_spectrum,
        fft_music_spectrum,

        fir_speech_spectrum,
        fir_music_spectrum,

        iir_speech_spectrum,
        iir_music_spectrum,

        # Overall analysis
        overall_waveform,
        overall_spectrum,

        # Performance
        results
    )


# ============================================================
# 15. HIGH-CONTRAST CSS
# ============================================================
#
# IMPORTANT:
#
# The background image is DARK.
#
# Therefore:
#   • Cards = LIGHT
#   • Text = DARK
#   • Headings = DARK BLUE
#   • Buttons = PURPLE/CYAN
#
# This prevents:
#   ❌ dark text on dark background
#   ❌ white text on white cards
#   ❌ transparent-looking text
#
# Native Gradio audio player is NOT globally modified.
# ============================================================

CSS = r"""

/* ==========================================================
   FULL PAGE
   ========================================================== */

html,
body {

    margin: 0 !important;
    padding: 0 !important;

    background-color: #080b25 !important;

}


/* ==========================================================
   BACKGROUND IMAGE
   ========================================================== */

body::before {

    content: "";

    position: fixed;

    inset: 0;

    z-index: -2;

    background-image:

        linear-gradient(
            rgba(5, 8, 30, 0.58),
            rgba(9, 8, 35, 0.68)
        ),

        url("https://i.ibb.co/9mMML0H9/background.png");

    background-size: cover;

    background-position: center;

    background-repeat: no-repeat;

}


/* ==========================================================
   GRADIO ROOT
   ========================================================== */

#root {

    background: transparent !important;

}


.gradio-container {

    max-width: 1450px !important;

    background: transparent !important;

}


/* ==========================================================
   HERO
   ========================================================== */

.hero {

    margin: 25px auto 30px auto;

    padding: 32px;

    text-align: center;

    border-radius: 26px;

    background:
        rgba(255,255,255,0.96);

    border:
        1px solid
        rgba(255,255,255,0.98);

    box-shadow:
        0 18px 45px
        rgba(0,0,0,0.38);

}


.hero-title {

    font-size: 40px;

    font-weight: 900;

    letter-spacing: 1px;

    color: #172554 !important;

    margin-bottom: 10px;

}


.hero-subtitle {

    color: #334155 !important;

    font-size: 17px;

    margin: 5px 0;

}


/* ==========================================================
   LIGHT CARDS
   ========================================================== */

.glass-card {

    background:
        rgba(248,250,252,0.97) !important;

    border:
        1px solid
        rgba(255,255,255,0.98) !important;

    border-radius:
        22px !important;

    padding:
        22px !important;

    box-shadow:
        0 14px 40px
        rgba(0,0,0,0.32) !important;

}


/* ==========================================================
   SECTION TITLES
   ========================================================== */

.section-title {

    color: #172554 !important;

    font-size: 21px;

    font-weight: 900;

    padding-bottom: 10px;

    margin-bottom: 18px;

    border-bottom:
        3px solid
        #c4b5fd;

}


/* ==========================================================
   MARKDOWN
   ========================================================== */

.glass-card .prose,
.glass-card .prose *,
.glass-card .markdown,
.glass-card .markdown * {

    color: #172033 !important;

}


.glass-card h1,
.glass-card h2,
.glass-card h3 {

    color: #172554 !important;

    font-weight: 850 !important;

}


.glass-card strong,
.glass-card b {

    color: #172554 !important;

}


.glass-card p {

    color: #334155 !important;

}


/* ==========================================================
   LABELS
   ========================================================== */

.glass-card label {

    color: #172554 !important;

    font-weight: 700 !important;

}


.glass-card .label-wrap,
.glass-card .label-wrap span {

    color: #172554 !important;

}


/* ==========================================================
   PROCESS BUTTON
   ========================================================== */

#process-btn {

    width: 100% !important;

    min-height: 60px !important;

    border: none !important;

    border-radius: 17px !important;

    color: #ffffff !important;

    font-size: 19px !important;

    font-weight: 900 !important;

    background:
        linear-gradient(
            90deg,
            #6d28d9,
            #7c3aed,
            #0891b2
        ) !important;

    box-shadow:
        0 10px 30px
        rgba(76,29,149,0.40) !important;

}


#process-btn:hover {

    box-shadow:
        0 14px 38px
        rgba(8,145,178,0.45) !important;

    transform:
        translateY(-2px);

}


/* ==========================================================
   RESULT PANEL
   ========================================================== */

#result-panel {

    margin-top: 25px;

    padding: 28px;

    border-radius: 24px;

    background:
        rgba(248,250,252,0.97) !important;

    border:
        1px solid
        rgba(255,255,255,0.98) !important;

    box-shadow:
        0 16px 42px
        rgba(0,0,0,0.35) !important;

}


.result-title {

    color: #172554 !important;

    font-size: 29px;

    font-weight: 900;

    margin-bottom: 5px;

}


.result-subtitle {

    color: #475569 !important;

    font-size: 15px;

    margin-bottom: 22px;

}


/* ==========================================================
   RESULT STATISTICS
   ========================================================== */

.stats-grid {

    display: grid;

    grid-template-columns:
        repeat(4, 1fr);

    gap: 14px;

}


.stat-card {

    background:
        linear-gradient(
            135deg,
            #eef2ff,
            #ecfeff
        );

    border:
        1px solid
        #c7d2fe;

    border-radius:
        16px;

    padding:
        18px;

}


.stat-label {

    color:
        #64748b !important;

    font-size:
        11px;

    font-weight:
        800;

    letter-spacing:
        1px;

}


.stat-value {

    color:
        #0f172a !important;

    font-size:
        21px;

    font-weight:
        900;

    margin-top:
        6px;

}


/* ==========================================================
   RESULT HEADINGS
   ========================================================== */

.result-heading {

    color:
        #4338ca !important;

    font-size:
        19px;

    font-weight:
        900;

    margin-top:
        25px;

    margin-bottom:
        12px;

}


/* ==========================================================
   CONFIGURATION
   ========================================================== */

.config-row {

    display:
        flex;

    flex-wrap:
        wrap;

    gap:
        12px;

}


.config-row span {

    background:
        #f1f5f9;

    color:
        #475569 !important;

    padding:
        10px 15px;

    border-radius:
        10px;

    border:
        1px solid
        #e2e8f0;

}


.config-row b {

    color:
        #0f172a !important;

    margin-left:
        5px;

}


/* ==========================================================
   METHOD ROW
   ========================================================== */

.method-row {

    display:
        grid;

    grid-template-columns:
        repeat(3, 1fr);

    gap:
        14px;

}


.method-item {

    padding:
        16px;

    border-radius:
        15px;

    border:
        1px solid
        #e2e8f0;

    background:
        #ffffff;

}


.method-item b {

    color:
        #172554 !important;

    font-size:
        17px;

}


.method-item small {

    display:
        block;

    color:
        #64748b !important;

    margin-top:
        6px;

}


.method-icon {

    margin-right:
        6px;

}


/* ==========================================================
   SUCCESS
   ========================================================== */

.success-box {

    margin-top:
        20px;

    padding:
        14px 18px;

    border-radius:
        12px;

    background:
        #ecfdf5;

    border:
        1px solid
        #86efac;

    color:
        #166534 !important;

}


/* ==========================================================
   AUDIO CARDS
   ========================================================== */

.audio-card {

    background:
        rgba(248,250,252,0.97) !important;

}


.audio-card label {

    color:
        #172554 !important;

}


/* ==========================================================
   ANALYSIS CARDS
   ========================================================== */

.analysis-card {

    background:
        rgba(248,250,252,0.97) !important;

    border-radius:
        22px !important;

    padding:
        18px !important;

    box-shadow:
        0 14px 38px
        rgba(0,0,0,0.30) !important;

}


.analysis-card label {

    color:
        #172554 !important;

    font-weight:
        800 !important;

}


/* ==========================================================
   GRAPH GRID
   ========================================================== */

.graph-card {

    background:
        rgba(248,250,252,0.97) !important;

    border:
        1px solid
        #e2e8f0 !important;

    border-radius:
        18px !important;

    padding:
        12px !important;

    box-shadow:
        0 10px 25px
        rgba(0,0,0,0.20) !important;

}


.graph-card label {

    color:
        #172554 !important;

    font-weight:
        850 !important;

}


/* ==========================================================
   PERFORMANCE
   ========================================================== */

.performance-heading {

    background:
        rgba(248,250,252,0.97);

    color:
        #172554 !important;

    padding:
        18px 22px;

    border-radius:
        18px 18px 0 0;

    font-size:
        22px;

    font-weight:
        900;

    border-bottom:
        3px solid
        #c4b5fd;

}


.performance-table {

    background:
        rgba(248,250,252,0.97) !important;

    border-radius:
        0 0 18px 18px !important;

    overflow:
        hidden !important;

    box-shadow:
        0 10px 30px
        rgba(0,0,0,0.25) !important;

}


/* ==========================================================
   FOOTER
   ========================================================== */

.footer {

    margin-top:
        25px;

    padding:
        25px;

    text-align:
        center;

    color:
        #e2e8f0 !important;

    background:
        rgba(15,23,42,0.82);

    border-radius:
        18px;

}


/* ==========================================================
   RESPONSIVE
   ========================================================== */

@media (max-width: 900px) {

    .stats-grid {

        grid-template-columns:
            repeat(2, 1fr);

    }

    .method-row {

        grid-template-columns:
            1fr;

    }

}


@media (max-width: 600px) {

    .hero-title {

        font-size:
            29px;

    }

    .stats-grid {

        grid-template-columns:
            1fr;

    }

}


/* ==========================================================
   DARK MODE OVERRIDE
   ========================================================== */

.dark {

    --body-background-fill:
        transparent !important;

}


/* ==========================================================
   KEEP OUR CARDS READABLE
   ========================================================== */

.glass-card,
#result-panel,
.analysis-card,
.graph-card,
.performance-heading {

    color:
        #172033 !important;

}

"""


# ============================================================
# 16. BUILD GUI
# ============================================================

with gr.Blocks(
    title="DSP Audio Separation Studio"
) as demo:


    # ========================================================
    # HEADER
    # ========================================================

    gr.HTML(
        """
        <div class="hero">

            <div class="hero-title">
                🎧 DSP AUDIO SEPARATION STUDIO
            </div>

            <div class="hero-subtitle">
                Speech & Music Separation using
                <b>FFT • FIR • IIR Filters</b>
            </div>

            <div class="hero-subtitle">
                Digital Signal Processing Analysis System
            </div>

        </div>
        """
    )


    # ========================================================
    # INPUT SECTION
    # ========================================================

    with gr.Row():


        # ----------------------------------------------------
        # AUDIO INPUT
        # ----------------------------------------------------

        with gr.Column(
            elem_classes=["glass-card"]
        ):

            gr.HTML(
                """
                <div class="section-title">
                    🎵 AUDIO INPUT
                </div>
                """
            )


            mixed_audio = gr.Audio(
                label="🎵 Mixed Audio — Speech + Music",
                type="filepath"
            )


            speech_reference = gr.Audio(
                label="🗣️ Reference Speech — Optional",
                type="filepath"
            )


            music_reference = gr.Audio(
                label="🎼 Reference Music — Optional",
                type="filepath"
            )


        # ----------------------------------------------------
        # DSP PARAMETERS
        # ----------------------------------------------------

        with gr.Column(
            elem_classes=["glass-card"]
        ):

            gr.HTML(
                """
                <div class="section-title">
                    ⚙️ DSP PARAMETERS
                </div>
                """
            )


            cutoff = gr.Slider(
                minimum=500,
                maximum=3000,
                value=1200,
                step=100,
                label="🎚️ Cutoff Frequency (Hz)"
            )


            fir_taps = gr.Slider(
                minimum=51,
                maximum=301,
                value=101,
                step=10,
                label="🔧 FIR Number of Taps"
            )


            fir_order = gr.Slider(
                minimum=50,
                maximum=300,
                value=100,
                step=10,
                label="📐 FIR Order"
            )


            iir_order = gr.Slider(
                minimum=2,
                maximum=10,
                value=6,
                step=1,
                label="🔵 IIR Butterworth Order"
            )


    # ========================================================
    # BUTTON
    # ========================================================

    gr.HTML("<br>")


    process_button = gr.Button(
        "🚀  START AUDIO SEPARATION",
        elem_id="process-btn"
    )


    # ========================================================
    # RESULT INFORMATION
    # ========================================================

    result_info = gr.HTML(
        """
        <div class="result-title">
            🎛️ SYSTEM READY
        </div>

        <div class="result-subtitle">
            Upload the mixed audio and press
            <b>START AUDIO SEPARATION</b>.
        </div>

        <div class="success-box">
            🟢 Ready to process FFT, FIR and IIR separation.
        </div>
        """,

        elem_id="result-panel"
    )


    # ========================================================
    # SEPARATED AUDIO OUTPUTS
    # ========================================================

    gr.HTML(
        """
        <div class="section-title"
             style="
             color:#ffffff !important;
             background:rgba(15,23,42,0.82);
             padding:15px 20px;
             border-radius:15px;
             ">
            🔊 SEPARATED AUDIO OUTPUTS
        </div>
        """
    )


    # ========================================================
    # THREE METHODS
    # ========================================================

    with gr.Row():


        # ----------------------------------------------------
        # FFT
        # ----------------------------------------------------

        with gr.Column(
            elem_classes=[
                "glass-card",
                "audio-card"
            ]
        ):

            gr.Markdown(
                """
                ## 🟣 FFT METHOD

                **Frequency-domain separation**

                Low frequency → **Speech**

                High frequency → **Music**
                """
            )


            fft_speech = gr.Audio(
                label="🗣️ FFT — Separated Speech",
                type="filepath",
                buttons=["download"]
            )


            fft_music = gr.Audio(
                label="🎼 FFT — Separated Music",
                type="filepath",
                buttons=["download"]
            )


        # ----------------------------------------------------
        # FIR
        # ----------------------------------------------------

        with gr.Column(
            elem_classes=[
                "glass-card",
                "audio-card"
            ]
        ):

            gr.Markdown(
                """
                ## 💜 FIR METHOD

                **Finite Impulse Response Filter**

                Low-pass → **Speech**

                High-pass → **Music**
                """
            )


            fir_speech = gr.Audio(
                label="🗣️ FIR — Separated Speech",
                type="filepath",
                buttons=["download"]
            )


            fir_music = gr.Audio(
                label="🎼 FIR — Separated Music",
                type="filepath",
                buttons=["download"]
            )


        # ----------------------------------------------------
        # IIR
        # ----------------------------------------------------

        with gr.Column(
            elem_classes=[
                "glass-card",
                "audio-card"
            ]
        ):

            gr.Markdown(
                """
                ## 🔵 IIR METHOD

                **Butterworth IIR Filter**

                Low-pass → **Speech**

                High-pass → **Music**
                """
            )


            iir_speech = gr.Audio(
                label="🗣️ IIR — Separated Speech",
                type="filepath",
                buttons=["download"]
            )


            iir_music = gr.Audio(
                label="🎼 IIR — Separated Music",
                type="filepath",
                buttons=["download"]
            )


    # ========================================================
    # SIGNAL ANALYSIS TITLE
    # ========================================================

    gr.HTML(
        """
        <div class="section-title"
             style="
             color:#ffffff !important;
             background:rgba(15,23,42,0.82);
             padding:15px 20px;
             border-radius:15px;
             margin-top:25px;
             ">
            📊 SIGNAL ANALYSIS
        </div>
        """
    )


    # ========================================================
    # INDIVIDUAL WAVEFORMS
    # ========================================================
    #
    # Each signal gets its own waveform graph.
    # ========================================================

    gr.HTML(
        """
        <div class="performance-heading">
            🎵 INDIVIDUAL TIME-DOMAIN WAVEFORMS
        </div>
        """
    )


    with gr.Row():

        with gr.Column(
            elem_classes=["graph-card"]
        ):

            mixed_waveform_output = gr.Image(
                label="📈 Mixed Input — Waveform"
            )


        with gr.Column(
            elem_classes=["graph-card"]
        ):

            fft_speech_waveform_output = gr.Image(
                label="🗣️ FFT Speech — Waveform"
            )


        with gr.Column(
            elem_classes=["graph-card"]
        ):

            fft_music_waveform_output = gr.Image(
                label="🎼 FFT Music — Waveform"
            )


    with gr.Row():

        with gr.Column(
            elem_classes=["graph-card"]
        ):

            fir_speech_waveform_output = gr.Image(
                label="🗣️ FIR Speech — Waveform"
            )


        with gr.Column(
            elem_classes=["graph-card"]
        ):

            fir_music_waveform_output = gr.Image(
                label="🎼 FIR Music — Waveform"
            )


        with gr.Column(
            elem_classes=["graph-card"]
        ):

            iir_speech_waveform_output = gr.Image(
                label="🗣️ IIR Speech — Waveform"
            )


    with gr.Row():

        with gr.Column(
            elem_classes=["graph-card"]
        ):

            iir_music_waveform_output = gr.Image(
                label="🎼 IIR Music — Waveform"
            )


    # ========================================================
    # INDIVIDUAL SPECTRA
    # ========================================================
    #
    # Each signal gets its own frequency-domain graph.
    # ========================================================

    gr.HTML(
        """
        <div class="performance-heading">
            📡 INDIVIDUAL FREQUENCY SPECTRA
        </div>
        """
    )


    with gr.Row():

        with gr.Column(
            elem_classes=["graph-card"]
        ):

            mixed_spectrum_output = gr.Image(
                label="📡 Mixed Input — Spectrum"
            )


        with gr.Column(
            elem_classes=["graph-card"]
        ):

            fft_speech_spectrum_output = gr.Image(
                label="🗣️ FFT Speech — Spectrum"
            )


        with gr.Column(
            elem_classes=["graph-card"]
        ):

            fft_music_spectrum_output = gr.Image(
                label="🎼 FFT Music — Spectrum"
            )


    with gr.Row():

        with gr.Column(
            elem_classes=["graph-card"]
        ):

            fir_speech_spectrum_output = gr.Image(
                label="🗣️ FIR Speech — Spectrum"
            )


        with gr.Column(
            elem_classes=["graph-card"]
        ):

            fir_music_spectrum_output = gr.Image(
                label="🎼 FIR Music — Spectrum"
            )


        with gr.Column(
            elem_classes=["graph-card"]
        ):

            iir_speech_spectrum_output = gr.Image(
                label="🗣️ IIR Speech — Spectrum"
            )


    with gr.Row():

        with gr.Column(
            elem_classes=["graph-card"]
        ):

            iir_music_spectrum_output = gr.Image(
                label="🎼 IIR Music — Spectrum"
            )


    # ========================================================
    # OVERALL ANALYSIS
    # ========================================================

    gr.HTML(
        """
        <div class="performance-heading">
            🔬 OVERALL ANALYSIS & COMPARISON
        </div>
        """
    )


    with gr.Row():

        with gr.Column(
            elem_classes=["analysis-card"]
        ):

            overall_waveform_output = gr.Image(
                label="📈 Overall Waveform Comparison"
            )


        with gr.Column(
            elem_classes=["analysis-card"]
        ):

            overall_spectrum_output = gr.Image(
                label="📡 Overall FFT Spectrum Comparison"
            )


    # ========================================================
    # PERFORMANCE EVALUATION
    # ========================================================

    gr.HTML(
        """
        <div class="performance-heading">
            📊 PERFORMANCE EVALUATION
        </div>
        """
    )


    performance_table = gr.Dataframe(
        headers=[
            "Method",
            "Speech Correlation",
            "Music Correlation",
            "Speech SNR (dB)",
            "Music SNR (dB)",
            "Speech RMSE",
            "Music RMSE"
        ],
        interactive=False,
        elem_classes=["performance-table"]
    )


    # ========================================================
    # FOOTER
    # ========================================================

    gr.HTML(
        """
        <div class="footer">

            🎧 <b>DSP Audio Separation Studio</b>

            <br><br>

            FFT &nbsp; • &nbsp;
            FIR &nbsp; • &nbsp;
            IIR &nbsp; • &nbsp;
            Waveform Analysis &nbsp; • &nbsp;
            Spectrum Analysis

            <br><br>

            Digital Signal Processing Mini Project

        </div>
        """
    )


    # ========================================================
    # BUTTON EVENT
    # ========================================================

    process_button.click(

        fn=process_audio,

        inputs=[

            mixed_audio,
            speech_reference,
            music_reference,

            cutoff,

            fir_taps,
            fir_order,
            iir_order
        ],

        outputs=[

            # Result information
            result_info,

            # Audio outputs
            fft_speech,
            fft_music,

            fir_speech,
            fir_music,

            iir_speech,
            iir_music,

            # Individual waveforms
            mixed_waveform_output,
            fft_speech_waveform_output,
            fft_music_waveform_output,

            fir_speech_waveform_output,
            fir_music_waveform_output,

            iir_speech_waveform_output,
            iir_music_waveform_output,

            # Individual spectra
            mixed_spectrum_output,
            fft_speech_spectrum_output,
            fft_music_spectrum_output,

            fir_speech_spectrum_output,
            fir_music_spectrum_output,

            iir_speech_spectrum_output,
            iir_music_spectrum_output,

            # Overall analysis
            overall_waveform_output,
            overall_spectrum_output,

            # Performance
            performance_table

        ]
    )


# ============================================================
# 17. LAUNCH
# ============================================================

demo.launch(
    share=True,
    css=CSS
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://615782085ace92f863.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
